In [ ]:
import numpy as np
import h5py
from holotomocupy.config import parse_args
from holotomocupy.reader import *
from holotomocupy.utils import *
from scipy.fft import fftn, ifftn, fftshift, fft2, ifft2
import scipy.ndimage as ndimage 

In [ ]:
args = parse_args('config1.conf')
read_acquisition_pars(args)

In [ ]:
def _draw_frame_edges_inplace(cube, p1, p2):
    """Draw edges into cube (assumes cube already zero)."""
    # Edges parallel to x
    cube[p1:p2, p1, p1] = 1; cube[p1:p2, p1, p2] = 1
    cube[p1:p2, p2, p1] = 1; cube[p1:p2, p2, p2] = 1
    # Edges parallel to y
    cube[p1, p1:p2, p1] = 1; cube[p1, p1:p2, p2] = 1
    cube[p2, p1:p2, p1] = 1; cube[p2, p1:p2, p2] = 1
    # Edges parallel to z
    cube[p1, p1, p1:p2] = 1; cube[p1, p2, p1:p2] = 1
    cube[p2, p1, p1:p2] = 1; cube[p2, p2, p1:p2] = 1

def rotate3d_once(vol, ang_xy_deg=28, ang_xz_deg=45, order=1):
    # angles in radians
    a = np.deg2rad(ang_xy_deg)  # rotation in (0,1) plane = about z-axis
    b = np.deg2rad(ang_xz_deg)  # rotation in (0,2) plane = about y-axis

    Rz = np.array([[ np.cos(a), -np.sin(a), 0],
                   [ np.sin(a),  np.cos(a), 0],
                   [ 0,          0,         1]], dtype=np.float64)

    Ry = np.array([[ np.cos(b), 0, np.sin(b)],
                   [ 0,         1, 0        ],
                   [-np.sin(b), 0, np.cos(b)]], dtype=np.float64)

    R = Ry @ Rz  # apply z-rot then y-rot (matches sequential calls order)

    # affine_transform uses inverse mapping:
    A = np.linalg.inv(R)

    center = (np.array(vol.shape) - 1) / 2.0
    offset = center - A @ center

    return ndimage.affine_transform(
        vol, A, offset=offset, order=order, mode="constant", cval=0.0, prefilter=(order > 1)
    )

# usage:
# obj = rotate3d_once(obj, 28, 45, order=1)
def gen_object(n, delta, beta):
    obj = np.zeros((n, n, n), dtype=np.float32)

    rr = (np.ones(8) * n * 0.2).astype(np.int32)
    amps = np.array([3, -3, 1, 3, -4, 1, 4], dtype=np.float32)
    dil  = (np.array([33, 28, 25, 21, 16, 10, 3], dtype=np.float32) / 256.0) * n

    # Precompute r^2 once
    ax = np.arange(-n//2, n//2, dtype=np.float32)
    x, y, z = np.meshgrid(ax, ax, ax, indexing="ij")
    r2 = x*x + y*y + z*z
    del x, y, z  # free memory

    # Precompute fcirc (complex64) once
    fcirc_list = []
    for d in dil:
        circ = (r2 < (d*d)).astype(np.float32, copy=False)
        fcirc_list.append(fftn(fftshift(circ),workers=-1).astype(np.complex64, copy=False))
    # (optional) free circ/r2 if tight on memory; r2 still needed nowhere after this
    # del r2

    # Precompute fcube for each kk (depends on r)
    cube = np.zeros((n, n, n), dtype=np.float32)  # reused
    fcube_list = []
    for kk in range(len(amps)):
        cube.fill(0.0)
        r = int(rr[kk])
        p1 = n//2 - r//2
        p2 = n//2 + r//2
        _draw_frame_edges_inplace(cube, p1, p2)
        fcube_list.append(fftn(fftshift(cube),workers=-1).astype(np.complex64, copy=False))

    # Reused complex work buffer for product in Fourier space
    work = np.empty((n, n, n), dtype=np.complex64)

    for kk, a in enumerate(amps):
        # work = fcube * fcirc
        np.multiply(fcube_list[kk], fcirc_list[kk], out=work)

        conv = fftshift(ifftn(work,workers=-1)).real  # float64->float32-ish; real view is float32? (depends)
        obj += a * (conv > 1.0)  # boolean -> float32 via multiplication; avoids an extra astype

    # Rotations/translations (these are often the dominant cost after FFTs)
    obj = rotate3d_once(obj, 28, 45, order=1)
    obj = np.roll(obj, -15*n//256, axis=2)
    obj = np.roll(obj, -10*n//256, axis=1)

    np.maximum(obj, 0, out=obj)
    v = (np.arange(-n//2, n//2, dtype=np.float32) / n)
    vx, vy, vz = np.meshgrid(v, v, v, indexing="ij")
    filt = fftshift(np.exp(-3.0 * (vx*vx + vy*vy + vz*vz)).astype(np.float32))

    fu = fftn((obj))
    obj = ifftn((fu * filt)).real

    obj[obj < 0] = 0
    return (obj * (-delta + 1j*beta)).astype(np.complex64, copy=False)

obj = gen_object(args.nobj//4,1,1e-2)
mshow_complex(obj[:, args.nobj//4//2],True)
mshow_complex(obj[args.nobj//4//2],True)
np.save('/data2/tmp/obj2.npy',obj)


In [ ]:
import scipy.ndimage as ndimage
with h5py.File(args.in_file) as fid:        
    pos = ndimage.zoom((fid[f'/exchange/cshifts_final'][:,:args.ndist]).astype('float32')/4,[4,1,1])      
    

!wget -nc https://g-110014.fd635.8443.data.globus.org/holotomocupy/examples_synthetic/data/prb_id16a/prb_abs_2048.tiff -P data/prb_id16a
!wget -nc https://g-110014.fd635.8443.data.globus.org/holotomocupy/examples_synthetic/data/prb_id16a/prb_phase_2048.tiff -P data/prb_id16a

prb_abs = read_tiff(f'data/prb_id16a/prb_abs_2048.tiff')[:args.ndist]
prb_phase = read_tiff(f'data/prb_id16a/prb_phase_2048.tiff')[:args.ndist]
prb = prb_abs*np.exp(1j*prb_phase).astype('complex64')
prb = prb[:,512:-512,512:-512]
prb = prb[:, ::2]+prb[:, 1::2]
prb = prb[:, :, ::2]+prb[:, :, 1::2]/4
n = prb.shape[-1]
v = (np.arange(-n//2, n//2, dtype=np.float32) / n)
vx, vy = np.meshgrid(v, v, indexing="ij")
filt = fftshift(np.exp(-4.0 * (vx*vx + vy*vy)).astype(np.float32))

fu = fft2((prb))
prb = ifft2((fu *filt))

prb /= np.mean(np.abs(prb),axis=(1,2))[:,None,None]

with h5py.File('/data2/vnikitin/syn_sample.h5', 'w') as fid:
    fid.create_dataset('/exchange/z1', data=args.z1, dtype='float32')
    fid.create_dataset('/exchange/detector_pixelsize', data=[args.detector_pixelsize*4], dtype='float32')
    fid.create_dataset('/exchange/energy', data=[args.energy], dtype='float32')
    fid.create_dataset('/exchange/focusdetectordistance', data=[args.focustodetectordistance], dtype='float32')    
    fid.create_dataset('/exchange/cshifts_final', data=pos, dtype='float32')    
    fid.create_dataset('/exchange/cshifts_error', data=pos+((np.random.random(pos.shape)-0.5)*4).astype('float32'), dtype='float32')    
    fid.create_dataset('/exchange/prb', data = prb)    
    fid.create_dataset('/exchange/obj', data = obj)    
    fid.create_dataset('/exchange/theta', data = np.linspace(0,180,18000).astype('float32')[:,None]) 